# 01. Data and panel construction

**What this notebook establishes.** Where the numbers in the Study data paragraph come
from, and why the panel is built the way it is.

The two decisions that matter and are easy to get wrong:

1. **The universe is the birth series, not the death series.** A region-year with births
   and no deaths is an observed zero. The archived earlier version of this project keyed
   the panel on deaths, which silently dropped 21,128 municipality-years (34% of the
   total) — all of them zero-death, and concentrated in exactly the small units the
   study is about. That is informative missingness and it biased the reliability tiers.

2. **The cause grouping was inherited, not re-derived.** The four action groups come
   from the source cause panel. This notebook can verify the totals but cannot verify
   the ICD-10 mapping, and the manuscript says so.

In [1]:
import json
from pathlib import Path

import arviz as az
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC, MODEL, RES = ROOT / "data/processed", ROOT / "2-model", ROOT / "3-results"

def check(label, computed, published, tol=1e-9):
    "Recompute a published value and fail loudly if the manuscript no longer matches."
    ok = abs(float(computed) - float(published)) <= tol
    print(f"{'OK  ' if ok else 'MISMATCH'}  {label}: manuscript={published}  recomputed={computed}")
    assert ok, f"{label}: manuscript says {published}, artefacts say {computed}"

def classified(draws, lo_q=0.025, hi_q=0.975):
    "The single criterion used everywhere in this study: 95% ETI excluding zero."
    lo, hi = np.quantile(draws, lo_q, axis=1), np.quantile(draws, hi_q, axis=1)
    return (lo > 0) | (hi < 0)

def slopes(idata, var="b"):
    return idata.posterior[var].stack(sample=("chain", "draw")).values

In [2]:
panel = pd.read_csv(PROC / "panel_region_year.csv")
muni  = pd.read_csv(PROC / "panel_muni_year.csv")

print(f"region panel : {len(panel):,} rows, {panel.rgi_id.nunique()} regions, "
      f"{panel.UF.nunique()} states, {panel.year.min()}-{panel.year.max()}")
print(f"muni panel   : {len(muni):,} rows, {muni.CODMUNRES.nunique():,} municipalities")
print(f"zero-death municipality-years retained: {(muni.deaths_total == 0).sum():,} "
      f"({(muni.deaths_total == 0).mean():.1%})")

region panel : 5,610 rows, 510 regions, 27 states, 2014-2024
muni panel   : 61,435 rows, 5,593 municipalities
zero-death municipality-years retained: 21,128 (34.4%)


## The published totals

Every figure below appears in the Study data paragraph of the manuscript.

In [3]:
check("neonatal deaths", panel.deaths_total.sum(), 260023)
check("live births", panel.births.sum(), 30462542)
check("avoidable deaths", panel.avoidable.sum(), 117578)
check("immediate regions", panel.rgi_id.nunique(), 510)
check("states", panel.UF.nunique(), 27)
check("panel rows", len(panel), 5610)

OK    neonatal deaths: manuscript=260023  recomputed=260023
OK    live births: manuscript=30462542  recomputed=30462542
OK    avoidable deaths: manuscript=117578  recomputed=117578
OK    immediate regions: manuscript=510  recomputed=510
OK    states: manuscript=27  recomputed=27
OK    panel rows: manuscript=5610  recomputed=5610


## Exposure over the decade

Two facts carry weight later. Avoidable mortality per birth fell substantially while
all-cause neonatal mortality barely moved, and the number of births — the exposure that
every unit's precision depends on — fell by a fifth.

In [4]:
by_year = panel.groupby("year").agg(births=("births", "sum"),
                                    avoidable=("avoidable", "sum"),
                                    deaths=("deaths_total", "sum"))
by_year["avoidable_per_1000"] = (1000 * by_year.avoidable / by_year.births).round(2)
by_year["nmr_per_1000"] = (1000 * by_year.deaths / by_year.births).round(2)
display(by_year)

check("avoidable per 1000, 2014", by_year.avoidable_per_1000.iloc[0], 4.25)
check("avoidable per 1000, 2024", by_year.avoidable_per_1000.iloc[-1], 3.44)
check("NMR 2014", by_year.nmr_per_1000.iloc[0], 8.89)
check("NMR 2024", by_year.nmr_per_1000.iloc[-1], 8.32)
check("births change (%)", round(100 * (by_year.births.iloc[-1] / by_year.births.iloc[0] - 1), 1), -20.0)

,births,avoidable,deaths,avoidable_per_1000,nmr_per_1000
year,,,,,
2014,2979133,12662,26471,4.25,8.89
2015,3017563,12558,26309,4.16,8.72
2016,2857704,11745,24949,4.11,8.73
2017,2923441,11776,25446,4.03,8.70
2018,2944826,11303,24985,3.84,8.48
2019,2849064,10787,24296,3.79,8.53
2020,2730050,10167,22387,3.72,8.20
2021,2677008,9962,22318,3.72,8.34
2022,2561858,9336,21565,3.64,8.42


OK    avoidable per 1000, 2014: manuscript=4.25  recomputed=4.25
OK    avoidable per 1000, 2024: manuscript=3.44  recomputed=3.44
OK    NMR 2014: manuscript=8.89  recomputed=8.89
OK    NMR 2024: manuscript=8.32  recomputed=8.32
OK    births change (%): manuscript=-20.0  recomputed=-20.0


## Exposure per unit, which is what precision depends on

The median immediate region accumulated 115 avoidable deaths across the whole decade.
The median municipality accumulated 7. No statistical method creates events that did
not occur, so these two numbers largely determine everything that follows.

In [5]:
for label, frame, key in [("immediate region", panel, "rgi_id"),
                          ("municipality", muni.dropna(subset=["rgi_id"]), "CODMUNRES")]:
    pooled = frame.groupby(key).avoidable.sum()
    print(f"{label:18} n={len(pooled):>5,}  median={int(pooled.median()):>5}  "
          f">=50: {(pooled >= 50).sum():>4}   >=400: {(pooled >= 400).sum():>4}")

immediate region   n=  510  median=  115  >=50:  426   >=400:   56
municipality       n=5,570  median=    7  >=50:  431   >=400:   23


## Crosswalk loss, disclosed in Methods

23 municipality codes did not match the region crosswalk. They carry no deaths, so no
outcome is lost; 980 births are.

In [6]:
report = (PROC / "panel_build_report.txt").read_text()
print("\n".join(l for l in report.splitlines() if "unmatched" in l.lower() or "CROSSWALK" in l))
check("births lost to the crosswalk", muni.births.sum() - panel.births.sum(), 980)

CROSSWALK LOSSES
  municipality-years unmatched: 166
  deaths in unmatched rows    : 0
  municipalities unmatched    : 23
OK    births lost to the crosswalk: manuscript=980  recomputed=980
